# tcal v0003 against v0004 on the 16/17 July night

`tcal` v0004 is v0003 with one change: the noise-source temperature `t_ns_k` is the bench measurement (`noise_source` v0001) converted to the field receiver, instead of an assumed 917 K. Every other array in the two products is identical. This notebook shows what that does to the night memo 004 uses (21:30–07:55 MDT).

Both versions are read through `load_bundle`, with antenna integrations weighted by integration time and the `flags@v3` RFI bits and int32 wraps at zero weight, as in memo 004 (`make_products.night_spectra`). Until the `noise-source` PRs merge, the code comes from the branch worktrees (first cell).

In [ ]:
import os, sys
# Branch code until the noise-source PRs merge: eigsep_data and eigsep_cal from their worktrees,
# and memo 004's products script for the night's rows and weights.
WT = "/mnt/data02/eigsep-worktrees"
for p in (f"{WT}/eigsep_data-noise-source/src", f"{WT}/eigsep_cal-noise-source/src"):
    sys.path.insert(0, p)
MEMO = f"{WT}/memo-004-methods/memos/memo-004-marjum-2026-07-calibration"
sys.path.insert(0, MEMO)
os.environ.setdefault("EIGSEP_CAMPAIGN_ROOT", "/mnt/data02/eigsep/marjum-2026-07")
os.environ.setdefault("EIGSEP_SIBLINGS", "/mnt/data02/eigsep")   # make_products needs one; its commit stamps are not used here
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import make_products as mp
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
import eigsep_data, eigsep_cal
print(eigsep_data.__file__); print(eigsep_cal.__file__)

In [ ]:
# The night through both versions (a few minutes)
N = {}
for v in ("v0003", "v0004"):
    mp.TCAL_VERSION = v
    N[v] = mp.night_spectra()
f = N["v0004"]["f"]
assert np.array_equal(N["v0003"]["t"], N["v0004"]["t"])
z4 = np.load(mp.CAMPAIGN / "derived/tcal/v0004/solutions.npz")
z3 = np.load(mp.CAMPAIGN / "derived/tcal/v0003/solutions.npz")
ns = mp.noise_source.load("v0001", root=mp.CAMPAIGN)
t_ns4 = np.asarray(z4["t_ns_k"], float)
t_ns3 = np.full(f.size, float(z3["t_ns_k"]))
w = N["v0004"]["w"]
C3, C4 = "0.55", "k"
k = mp.band(f, (40, 200.01))
fm = (f >= 87.5) & (f <= 108.0)
b = mp.band(f, (50, 200.01)) & ~fm
print(f"{N['v0004']['ant'].sum()} antenna integrations; {f.size} channels")

## 1. The noise-source temperature each version applies

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(f[k], t_ns3[k], color=C3, label="v0003: assumed 917 K")
ax.plot(f[k], t_ns4[k], color=C4, label="v0004: bench, as the field receiver takes it")
ax.plot(f[k], ns.excess_smooth_k[k], ":", color=mp.C[0], label="bench excess at P (noise_source v0001)")
ax.set(ylabel="$T_\\mathrm{NS}$ (K)", ylim=(0, 1000), xlim=(40, 200), xlabel="frequency (MHz)")
ax.legend(loc="lower left", fontsize=7)
plt.tight_layout(); plt.show()
print(f"v0004 T_NS over 50-200 MHz: {np.nanmin(t_ns4[b]):.0f}-{np.nanmax(t_ns4[b]):.0f} K; "
      f"ratio to 917 K {np.nanmin((t_ns4/t_ns3)[b]):.3f}-{np.nanmax((t_ns4/t_ns3)[b]):.3f}")

**Figure 1.** `t_ns_k` of each product. v0003 takes a flat 917 K, the top of the NC302L's specified ENR range behind the 30 dB pad. v0004 takes the bench-measured excess at P (dotted, the cubic fit stored in `noise_source` v0001), converted with $T_\mathrm{NS} = M_\mathrm{NS}(T_\mathrm{load}+T_\mathrm{ex}) - M_\mathrm{load}T_\mathrm{load}$ for the field receiver (black). The two nearly overlay because the noise source is well matched ($M_\mathrm{NS}\approx 1$).

## 2. The night's calibrated spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
for v, c in (("v0003", C3), ("v0004", C4)):
    ax.semilogy(f[k], N[v]["tcal"][k], color=c, lw=0.8, label=f"tcal {v}")
ax.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set(ylabel="calibrated T at P (K)", xlabel="frequency (MHz)", xlim=(40, 200))
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

**Figure 2.** The night's calibrated antenna temperature at plane P, weighted mean over antenna integrations, for each version. Gaps are channels with less than half their integration time left after flagging (mostly the FM band). v0004 is lower almost everywhere; above ~172 MHz the two cross, and v0003's dip near 180 MHz is gone.

## 3. Through the night

In [ ]:
t = N["v0004"]["t"]
ant = N["v0004"]["ant"]
edges = np.arange(t[ant].min(), t[ant].max() + 600, 600.0)
cen = 0.5 * (edges[1:] + edges[:-1])
ib = np.digitize(t, edges) - 1
# GSM2016 x HFSS directivity (300 K ground), as in memo 004's Figure 5, at each bin's centre:
# the mean over the HFSS frequencies inside each band
hb = np.load(mp.HFSS_BEAM)
BANDS = ((60, 80), (130, 150))
fh = np.array([x for x in hb["freqs"] if any(lo <= x <= hi for lo, hi in BANDS)], float)
tsim, _, _ = mp.gsm_sim(cen, fh)
fig, ax = plt.subplots(figsize=(7, 4.2))
for (lo, hi), mk in zip(BANDS, ("o", "s")):
    bb = mp.band(f, (lo, hi))
    for v, c in (("v0003", C3), ("v0004", C4)):
        rows, ww = N[v]["cal_rows"][:, bb], w[:, bb]
        y = np.array([mp.wmean(rows[ib == j].ravel(), ww[ib == j].ravel()) if ww[ib == j].sum() > 0 else np.nan
                      for j in range(cen.size)])
        ax.semilogy(mp.as_num(cen), y, mk + "-", ms=2, lw=0.6, color=c, label=f"{v}, {lo}–{hi} MHz")
    sel = (fh >= lo) & (fh <= hi)
    ax.semilogy(mp.as_num(cen), tsim[:, sel].mean(1), "--", lw=1.2, color=mp.C[0],
                label=f"GSM × HFSS, {lo}–{hi} MHz ({sel.sum()} HFSS freqs)")
ax.set(ylabel="T (K)", xlabel="local time (MDT, UTC−6)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=mp.TZ))
ax.legend(ncol=3, fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.tight_layout(); plt.show()

**Figure 3.** Band means of the calibrated temperature in 10-minute bins (weighted over the bins' antenna integrations and channels), 60–80 and 130–150 MHz, with the GSM2016 × HFSS directivity prediction (dashed blue: zenith pointing, 300 K ground filling the lower hemisphere, as in memo 004's Figure 5) at each bin's centre, averaged over the HFSS frequencies in the band. The time structure is the same in both versions and only the scale moves: v0004 applies one $T_\mathrm{NS}$ spectrum to every integration, and the interpolation of the calibration visits is unchanged. The rise and fall is the sky: the Galactic plane through Cygnus passes near the zenith at this latitude. The data swing further than the prediction and peak earlier (at 70 MHz near LST 19.9 h against 21.4 h); the beam, the ground term or a calibration drift through the night could each contribute, and none is tested here. The simulation is at the antenna terminals and the data at plane P, so their levels are not like for like. The spikes near 22:40–23:00 MDT at 130–150 MHz are in both versions; they are residual RFI left by the flags, not calibration. The gap is a stretch with no calibrated rows.

## 4. Spectra through the night

In [ ]:
# Five 10-minute windows at evenly spaced times across the night's calibrated antenna integrations
# (tcal v0004); a window that lands where nothing is calibrated moves to the nearest one that has data.
from datetime import datetime
NSPEC, WIN_S = 5, 600.0
cal4, w4 = N["v0004"]["cal_rows"], N["v0004"]["w"]
# calibrated rows only: tcal is NaN outside its coverage (from 22:11 MDT on this night)
has = ant & (w4.sum(1) > 0) & np.isfinite(cal4[:, k]).any(1)
t_lo, t_hi = t[has].min() + WIN_S / 2, t[has].max() - WIN_S / 2
targets = np.linspace(t_lo, t_hi, NSPEC)
centres = np.arange(t_lo, t_hi + 1, 60.0)
n_rows = np.array([(has & (np.abs(t - c) <= WIN_S / 2)).sum() for c in centres])
# Spectral, skipping its pale centre (~0.4-0.65), which barely shows on white
cols = plt.cm.Spectral(np.r_[np.linspace(0, 0.33, 3), np.linspace(0.75, 1.0, NSPEC - 3)])
ok = centres[n_rows > 0]
wins = np.array([ok[np.argmin(np.abs(ok - tc))] for tc in targets])
# GSM2016 x HFSS directivity (300 K ground) at each window's centre, on the HFSS frequencies
fh_all = hb["freqs"][(hb["freqs"] >= 40) & (hb["freqs"] <= 200)].astype(float)
tsim_w, _, _ = mp.gsm_sim(wins, fh_all)
fig, ax = plt.subplots(figsize=(7, 4.2))
for j, (c, col) in enumerate(zip(wins, cols)):
    m = has & (np.abs(t - c) <= WIN_S / 2)
    spec = mp.wmean(cal4[m], w4[m])
    lab = f"{datetime.fromtimestamp(c, mp.TZ):%H:%M} MDT ({m.sum()} integrations)"
    ax.semilogy(f[k], spec[k], color=col, lw=0.8, label=lab)
    ax.semilogy(fh_all, tsim_w[j], "--", color=col, lw=0.9)
ax.plot([], [], "k-", lw=0.8, label="data (solid)")
ax.plot([], [], "k--", lw=0.9, label="GSM × HFSS (dashed)")
ax.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set(ylabel="calibrated T at P (K)", xlabel="frequency (MHz)", xlim=(40, 200))
ax.legend(fontsize=6.5, title="window centre (10 min)", title_fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

**Figure 4.** The calibrated antenna temperature at plane P (tcal v0004) in five 10-minute windows at evenly spaced times across the night's calibrated stretch (tcal covers this night from 22:11 MDT), each the weighted mean over that window's antenna integrations (integration time, with `flags@v3` RFI bits and int32 wraps at zero weight, and the channels memo 004 drops for having under half their time left blank). Each curve is an average of the integrations in its window, not a single integration. The dashed line of the same colour is the GSM2016 × HFSS directivity prediction at that window's centre (zenith pointing, 300 K ground, as in memo 004's Figure 5) on the HFSS frequencies; it is at the antenna terminals and the data at plane P, so their levels are not like for like. The spectrum rises through the first half of the night and falls through the second, most at low frequency, which is the rise and fall of Figure 3 across the band. The narrow line near 136 MHz in the 00:40 window is residual RFI the flags leave.